In [1]:
import kagglehub
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import pickle

## Prepare directory

In [2]:
# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
# Fix for displaying the minus sign '-' correctly in plots
plt.rcParams['axes.unicode_minus'] = False 

# --- 1. Create a directory for storing charts ---
# Define an output directory name
output_dir = "output_charts"
# Check if the directory exists, and create it if it doesn't
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"已创建文件夹: {output_dir}")


已创建文件夹: output_charts


## Download dataset

In [ ]:
try:
    print("Downloading dataset from Kaggle Hub...")
    # kagglehub.dataset_download returns the path to the extracted dataset folder
    path = kagglehub.dataset_download("tencars/392-crypto-currency-pairs-at-minute-resolution")
    print(f"Dataset downloaded and extracted to: {path}")
except Exception as e:
    print(f"Failed to download dataset. Please check your internet connection and Kaggle API authentication. Error: {e}")
    path = "./tencars-392-crypto-currency-pairs-at-minute-resolution" 
    if not os.path.exists(path):
        print("Local dataset path not found, the script cannot continue.")
        exit()
    print(f"Using local cache path: {path}")

100%|██████████| 1.86G/1.86G [01:51<00:00, 18.0MB/s]

Extracting files...


Dataset downloaded and extracted to: /Users/ngtzekean/.cache/kagglehub/datasets/tencars/392-crypto-currency-pairs-at-minute-resolution/versions/1231


In [ ]:
def load_and_preprocess_with_duckdb(file_path, resample_rule='1h'):
    """
    Loads data from a CSV file using DuckDB, and performs resampling and preprocessing.
    """
    coin_name = os.path.basename(file_path).replace('.csv', '')
    print(f"\n>>>>> Starting to process with DuckDB: {coin_name.upper()} <<<<<")

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None

    # DuckDB SQL Query
    # Core idea:
    # 1. read_csv_auto('{file_path}'): Directly reads data from the CSV file.
    # 2. to_timestamp(time / 1000): Converts Unix timestamp (in milliseconds) to DuckDB's timestamp type.
    # 3. date_trunc('hour', ...): This is DuckDB's way to resample by hour. It truncates the time to the beginning of the hour.
    # 4. GROUP BY time_bucket: Groups rows by the truncated time bucket.
    # 5. arg_max/arg_min: A clever trick to get the first/last value within a group.
    #    - arg_max(close, to_timestamp(time / 1000)) gets the 'close' value from the row with the latest timestamp in the group (i.e., 'last').
    #    - arg_min(open, to_timestamp(time / 1000)) gets the 'open' value from the row with the earliest timestamp in the group (i.e., 'first').
    query = f"""
    SELECT
        date_trunc('hour', to_timestamp(time / 1000)) AS time_bucket,
        arg_min(open, to_timestamp(time / 1000)) AS open,
        max(high) AS high,
        min(low) AS low,
        arg_max(close, to_timestamp(time / 1000)) AS close,
        sum(volume) AS volume
    FROM read_csv_auto(
        '{file_path}', 
        header=true, 
        columns={'time': 'BIGINT', 'open': 'DOUBLE', 'high': 'DOUBLE', 'low': 'DOUBLE', 'close': 'DOUBLE', 'volume': 'DOUBLE'}
    )
    WHERE time > 0 -- A simple validity check
    GROUP BY time_bucket
    ORDER BY time_bucket;
    """
    
    print("Executing DuckDB query (handling millisecond timestamps)...")

    # Execute the query and convert the result directly to a Pandas DataFrame
    try:
        df_resampled = duckdb.query(query).to_df()
    except Exception as e:
        print(f"DuckDB query failed: {e}")
        return None

    # Set the time column as the index for easier subsequent operations
    df_resampled.set_index('time_bucket', inplace=True)
    
    print(f"Processing complete. Obtained {len(df_resampled)} hourly records.")
    
    return df_resampled

In [12]:
duckdb.execute(
"""
create or replace table crypto_data as (
    select *,'btc' as coin_name
    from read_csv_auto(
    '../data/crypto/btcusd.csv',
    header=true,
    columns={'time': 'BIGINT', 'open': 'DOUBLE', 'high': 'DOUBLE', 'low': 'DOUBLE', 'close': 'DOUBLE', 'volume': 'DOUBLE'}
    )

    union all

    select *,'eth' as coin_name
    from read_csv_auto(
    '../data/crypto/ethusd.csv',
    header=true,
    columns={'time': 'BIGINT', 'open': 'DOUBLE', 'high': 'DOUBLE', 'low': 'DOUBLE', 'close': 'DOUBLE', 'volume': 'DOUBLE'}
    )
)
""")

In [ ]:
duckdb.execute(
"""
insert into crypto_data values (
    select *,'eth' as coin_name
    from read_csv_auto(
    '../data/crypto/ethusd.csv',
    header=true,
    columns={'time': 'BIGINT', 'open': 'DOUBLE', 'high': 'DOUBLE', 'low': 'DOUBLE', 'close': 'DOUBLE', 'volume': 'DOUBLE'}
    )
)
""")

ParserException: Parser Error: syntax error at or near "select"

LINE 3:     select *,'btc' as coin_name
            ^

In [17]:
duckdb.query("select date_trunc('hour', to_timestamp(time / 1000)) AS time_bucket, coin_name, max(high), min(low) from crypto_data group by 1,2")

┌──────────────────────────┬───────────┬───────────────┬───────────────┐
│       time_bucket        │ coin_name │   max(high)   │   min(low)    │
│ timestamp with time zone │  varchar  │    double     │    double     │
├──────────────────────────┼───────────┼───────────────┼───────────────┤
│ 2016-12-28 22:00:00+08   │ btc       │        969.62 │        965.58 │
│ 2016-12-29 04:00:00+08   │ btc       │        974.64 │        965.62 │
│ 2016-12-29 05:00:00+08   │ btc       │        978.98 │         965.0 │
│ 2016-12-29 17:00:00+08   │ btc       │        979.91 │        974.94 │
│ 2016-12-30 17:00:00+08   │ btc       │        947.08 │        935.26 │
│ 2016-12-31 03:00:00+08   │ btc       │        962.29 │        957.85 │
│ 2017-01-01 00:00:00+08   │ btc       │        964.18 │         956.5 │
│ 2017-01-01 01:00:00+08   │ btc       │        967.99 │        961.21 │
│ 2017-01-01 12:00:00+08   │ btc       │         963.9 │        961.27 │
│ 2017-01-01 17:00:00+08   │ btc       │        966